# The Rest of LLMOps: 9 Things Beyond "Getting a Model Running"

Everything covered so far in this notebook series — file formats, hosting platforms,
serving frameworks, and local-to-production serving with Ollama/llama.cpp — answers
one question: **how do you get a model to respond to requests?** That's necessary but
not sufficient. A model that responds is not the same as a model that's safe, cheap,
debuggable, and trustworthy to run in production.

This notebook covers the **9 missing pieces** of the full LLMOps picture, each with why
it matters and which tools/frameworks are used to achieve it, as the landscape stands
in 2026.

**Contents**
1. Observability & monitoring
2. Security & access control
3. Cost optimization in production
4. CI/CD & MLOps integration
5. Evaluation before and after deployment
6. Caching strategies
7. Multi-model routing & gateways
8. Guardrails & content moderation
9. Licensing considerations
10. How it all fits together


## 1. Observability & Monitoring

### Why it matters
Once a model is serving real traffic, "it works on my machine" stops being useful.
Without observability you cannot answer basic operational questions: which requests
are slow, which are failing, what a user's multi-turn conversation actually looked
like, how much a given feature costs per day, or why output quality quietly degraded
after a prompt change. LLM calls are also **non-deterministic and expensive to
re-run**, which makes after-the-fact debugging much harder than with traditional
software — you need the trace captured the first time.

### What "observability" covers for an LLM app specifically
- **Tracing**: the full call graph of a request — retrieval steps, tool calls,
  intermediate reasoning, the final generation — not just a single request/response
  log line.
- **Cost & token tracking**: per-request and aggregate token usage and spend.
- **Latency breakdown**: time-to-first-token (TTFT) vs. total generation time, which
  matters enormously for perceived chat responsiveness.
- **Prompt versioning**: knowing exactly which prompt template produced which output.
- **Evaluation hooks**: attaching quality scores to traces (ties into Section 5).

### How to achieve it — the tools
- **Langfuse** — open source (MIT license), self-hostable, the most widely adopted
  open tracing/prompt-management tool; strong session-replay and multi-turn
  conversation reconstruction; native SDKs plus 50+ framework connectors.
- **LangSmith** — built by the LangChain team; the deepest native integration if
  you're building with LangChain/LangGraph specifically (this fits directly with
  [[langchain-langfuse-observability]] work); proprietary, usage-based pricing.
- **Arize Phoenix** — source-available (Elastic License 2.0), OpenTelemetry-native,
  strong built-in evaluation metrics (faithfulness, hallucination, drift detection)
  out of the box, given Arize's roots in classical ML monitoring.
- **MLflow Tracing** — fully open source (Apache 2.0), notable for bundling tracing,
  evaluation, and governance in one platform with no enterprise feature paywall —
  relevant given your existing MLflow use on the churn pipeline.
- **OpenTelemetry GenAI semantic conventions / OpenLLMetry / OpenInference** — a
  standardized trace format so instrumentation isn't locked to one vendor's SDK;
  tools built on this (Phoenix, OpenObserve) let you swap backends without
  re-instrumenting the application.
- **Helicone** — a lightweight, drop-in proxy-based option, useful when you want basic
  tracing and cost monitoring without adopting a full platform.

### Practical note
The real migration cost when changing observability tools isn't historical data — it's
**instrumentation**. Choosing an OpenTelemetry-based tool up front avoids being locked
into a proprietary SDK later.


## 2. Security & Access Control

### Why it matters
An exposed LLM endpoint is an exposed attack surface. Without access control, a public
endpoint can be scraped for free inference, abused to run up your GPU bill, or used as
an entry point for prompt injection attacks that manipulate the model into leaking
data or misusing connected tools. This risk compounds once an LLM has **tool-calling
or agentic capability**, since a successful injection can now trigger real actions, not
just a bad text response.

### What it covers
- **Authentication & API key management** — controlling who can call the endpoint at
  all, ideally via scoped, revocable keys rather than one shared secret.
- **Rate limiting & quota enforcement** — per-user or per-team request/spend caps to
  contain abuse and runaway cost.
- **Input validation & prompt injection defense** — detecting attempts to override
  system instructions, extract the system prompt, or smuggle malicious instructions
  through retrieved documents or tool outputs (**indirect** injection).
- **Network-level isolation** — VPC boundaries, firewall rules, and private endpoints
  so the model isn't reachable from the open internet unless it needs to be.

### How to achieve it — the tools
- **API gateways with built-in auth/rate limiting** (see Section 7) — LiteLLM's
  virtual keys and budgets, Portkey, or Kong AI Gateway all handle this at the
  infrastructure layer rather than in application code.
- **Llama Guard / Prompt Guard (Meta)** — lightweight classifier models specifically
  trained to detect unsafe prompts and jailbreak/injection attempts.
- **NeMo Guardrails (NVIDIA)** — a programmable rail system (using its Colang
  dialogue-flow language) that can intercept and block unsafe input *and* output
  patterns, including basic prompt-injection heuristics.
- **LLM Guard, Rebuff, Lakera Guard** — dedicated input-defense layers focused
  specifically on jailbreak/injection detection and PII masking before a request
  ever reaches the model.
- **Structural enforcement (often overlooked)** — scoped tool permissions and a hard
  gate on irreversible actions for any agent with tool access. Filtering the prompt
  *lowers the probability* of a bad output; capability-scoping *removes the
  possibility* of a bad action — the two are complementary, not substitutes.

### The threat taxonomy worth knowing
- **Direct injection** — a user's own input tries to override the system prompt.
- **Indirect injection** — malicious instructions arrive via an external source the
  model reads (a fetched webpage, a RAG document, a tool result) rather than the
  user's own message; input-only classifiers do not catch this — it needs
  retrieval/output-stage checks too.
- **System prompt leakage** — the model disclosing its own instructions in a
  response (OWASP LLM07) — defended mainly through prompt-template hardening and
  output filtering, not input classifiers.


## 3. Cost Optimization in Production

### Why it matters
The platform comparison in the earlier deployment notebook covers what's cheap to
**start on** — but ongoing GPU spend at real traffic volumes is where LLM hosting
costs actually spiral. A model that costs $0.002 to test can cost thousands of dollars
a month once it's serving production traffic, and the difference between an
optimized and unoptimized deployment is often a large multiple, not a rounding error.

### What drives cost, in priority order
1. **Model/token efficiency** — right-sizing the model for the task (a smaller
   fine-tuned or quantized model is often "good enough" and dramatically cheaper per
   token than defaulting to the largest available model).
2. **Caching** — avoiding redundant inference entirely for repeated or
   overlapping requests (expanded in Section 6).
3. **Batching** — serving engines like vLLM/SGLang batch concurrent requests to
   maximize GPU utilization per dollar; the choice of serving engine directly affects
   cost efficiency, not just raw speed.
4. **Routing** — sending cheaper/faster requests to smaller models and only escalating
   to larger models when needed (a gateway-level decision, Section 7).

### How to achieve it — the tools/techniques
- **Spot / preemptible GPU instances** — available on most cloud GPU providers at a
  fraction of on-demand pricing, suitable for batch or fault-tolerant workloads that
  can handle interruption.
- **Autoscaling to zero** — serverless platforms (RunPod, Modal, Beam — see the
  deployment services notebook) and Kubernetes-based autoscalers (e.g. KEDA) that
  scale GPU workers down to zero during idle periods, so you don't pay for idle
  capacity.
- **Quantization** — running a 4-bit/8-bit quantized model (GGUF, AWQ, GPTQ) instead
  of full precision cuts both memory footprint and, often, cost per token, at a
  measured quality trade-off (covered in the file-formats notebook).
- **LLM gateways with budget controls** — LiteLLM's per-team/per-key spend caps,
  Portkey's cost tracking, or Merge Gateway's request-level cost attribution give
  visibility into *where* spend is going, which is a prerequisite for optimizing it.
- **Request-level model routing** — tools like RouteLLM or a gateway's routing rules
  send simple queries to a cheap/small model and only route complex queries to an
  expensive one.


## 4. CI/CD & MLOps Integration

### Why it matters
Deploying a model once is easy. Deploying it **repeatably, safely, and reversibly** —
the same discipline applied to any other production software — is what actually
prevents a bad prompt change or a bad fine-tune from silently degrading production
traffic. This is the same MLOps muscle used for classical ML (as in the
[[mlops-p2]] churn-prediction pipeline with MLflow/Docker/GCP), applied specifically
to LLM artifacts and prompts instead of scikit-learn models.

### What it covers
- **Model/prompt versioning** — treating prompt templates and model checkpoints as
  versioned artifacts, not ad-hoc edits.
- **Rollback strategy** — the ability to revert quickly to a known-good model version
  or prompt if a new deployment regresses quality.
- **Canary and blue-green deployments** — routing a small percentage of live traffic
  to a new model/prompt version before fully cutting over, so a regression is caught
  on a limited blast radius rather than all users at once.
- **Automated redeployment pipelines** — a code or config change (new prompt, new
  fine-tuned checkpoint) triggering an automatic build → eval-gate → deploy sequence,
  rather than a manual push.

### How to achieve it — the tools
- **MLflow Model Registry** — versions model artifacts and stages
  (staging/production/archived), extendable to LLM checkpoints and prompt artifacts;
  directly reusable experience from the churn pipeline.
- **KServe / Seldon Core** — Kubernetes-native model-serving platforms with built-in
  support for canary rollouts and traffic-splitting between model versions.
- **A serving-engine-level router** (e.g. the vLLM Production Stack's router, or an
  API gateway) — can implement the actual traffic split for canary testing between two
  running model versions.
- **Standard CI/CD tooling (GitHub Actions, GitLab CI, Jenkins)** — orchestrates the
  build → containerize → eval-gate → deploy pipeline; the LLM-specific addition is
  wiring an **evaluation step** into that pipeline as a deployment gate (Section 5),
  rather than relying on unit tests alone.


## 5. Evaluation Before and After Deployment

### Why it matters
Unlike traditional software, an LLM application can silently regress without any code
change at all — a provider quietly updates the underlying model, a knowledge base grows
and retrieval quality drifts, or a one-line prompt edit tanks faithfulness across half
of production traffic. "It looked fine in the demo" is not a sufficient quality bar
once real users are involved, and the failure mode is often invisible until a user
reports it.

### What it covers
- **Pre-deployment regression testing** — a versioned "golden dataset" of test cases
  run automatically before any prompt or model change ships, blocking the deploy on
  failure (an eval-gate, tying directly into Section 4's CI/CD pipeline).
- **RAG-specific evaluation** — scoring retrieval quality (context precision/recall)
  separately from generation quality (faithfulness, answer relevancy), since a bad
  answer can originate from either stage.
- **LLM-as-judge** — using a separate (often stronger) model to score outputs against
  a rubric at a scale human review can't match, used as a pipeline gate or a
  production shadow-scoring step.
- **A/B testing** — comparing two prompts or models on live traffic with real user
  outcomes, not just offline scores.
- **Red-teaming / adversarial testing** — proactively probing for jailbreaks and
  unsafe outputs before an adversarial user finds them in production.

### How to achieve it — the tools
- **DeepEval** — a pytest-style Python framework (Apache 2.0); the natural fit for a
  **CI-gate**: eval cases are written like unit tests, and a failing eval blocks the
  deploy the same way a failing test would.
- **Ragas** — the canonical open-source RAG metric implementation (faithfulness,
  context precision/recall); best paired with DeepEval or another CI tool rather than
  used standalone, since it's metrics-only rather than a full pipeline.
- **Promptfoo** — a CLI/YAML tool built for rapid side-by-side prompt and model
  comparison, and the strongest open-source red-teaming/adversarial test suite in this
  category; better suited to pre-production iteration than as a production CI gate.
- **Braintrust / LangSmith / Arize Phoenix** — platforms that close the loop between
  production tracing (Section 1) and evaluation, adding persistent result storage,
  dashboards, and deployment quality gates on top of the underlying metrics.

### A common 2026 pattern
Teams frequently combine **Promptfoo** for pre-production prompt/model iteration and
red-teaming, **DeepEval** as the automated CI gate, and **Ragas** pulled in
specifically when debugging a RAG pipeline's retrieval quality — rather than
standardizing on a single tool for every stage.


## 6. Caching Strategies

### Why it matters
A large share of real-world LLM traffic is **redundant** — the same question asked
different ways, the same system prompt repeated on every request, or overlapping
context in multi-turn conversations. Recomputing a full forward pass for content
that's already been processed wastes both latency and money. Caching is one of the
highest-leverage, lowest-effort cost optimizations available (see Section 3's cost
priority list).

### The two distinct kinds of caching
1. **Exact/semantic response caching** — if an incoming request is identical or
   *semantically equivalent* to a previous one, return the cached response instead of
   calling the model again. This operates at the **application/gateway level**.
2. **Prefix/KV-cache caching** — if an incoming request shares a **prefix** (the same
   system prompt, the same retrieved documents, the same conversation history so far)
   with a prior request, reuse the already-computed key-value attention cache for that
   shared portion instead of recomputing it. This operates at the **serving-engine
   level**.

### How to achieve it — the tools
- **Semantic caching via a gateway** — Portkey offers this built in; **GPTCache** is a
  standalone open-source library for the same purpose, embedding-matching incoming
  queries against a cache of prior request/response pairs.
- **Prefix caching at the engine level** — this is exactly what **SGLang's
  RadixAttention** does (see the serving-frameworks notebook): it caches shared-prefix
  KV computation across requests, which is why SGLang measurably outperforms vLLM on
  throughput specifically for multi-turn chat, RAG, and agent workloads with
  overlapping context. **vLLM** also supports automatic prefix caching as a
  configurable feature.
- **Edge caching** — Cloudflare AI Gateway offers zero-infrastructure edge caching and
  analytics sitting in front of whichever model backend you use.

### Practical note
Semantic caching gives the biggest win when many *different users* ask
*similar-meaning* questions (e.g. an FAQ-style support bot). Prefix caching gives the
biggest win when the *same session or same knowledge base* is reused across many
requests (e.g. a long agent conversation, or a RAG app querying the same document set
repeatedly). The two are complementary, not interchangeable.


## 7. Multi-Model Routing & Gateways

### Why it matters
Production LLM applications rarely rely on a single model from a single provider
forever. Without a routing/gateway layer, every provider switch, every fallback for an
outage, and every per-team cost allocation has to be hand-coded into the application
itself — a maintenance burden that grows with every new model added. A gateway puts
**one stable endpoint** in front of many backends, so the application code stays the
same while what's behind it changes.

### What it covers
- **Unified API access** — one OpenAI-compatible interface across 100+ providers/models
  instead of juggling separate SDKs.
- **Fallback & failover** — automatically retrying against a different provider or
  model if the primary one errors or times out.
- **Load balancing** — spreading requests across multiple replicas or providers.
- **Budget & spend governance** — per-team or per-key spend caps and cost attribution.
- **Centralized observability, caching, and guardrails** — often bundled into the
  gateway layer itself rather than each application reimplementing them (ties directly
  into Sections 1, 2, and 6).

### How to achieve it — the tools
- **LiteLLM** — the dominant open-source, self-hosted option; a proxy you deploy
  yourself (Docker) that normalizes 100+ providers behind one OpenAI-format endpoint,
  with virtual-key budgeting built in. Best for teams running self-hosted models (e.g.
  your own vLLM deployment) alongside cloud provider APIs. Note: it's a routing/access
  layer, not a full control plane — no native guardrails or prompt versioning out of
  the box, and Python overhead can add latency above roughly 500 requests/second on a
  single instance.
- **Portkey** — a managed "control plane for AI traffic" adding semantic caching,
  guardrails, and governance on top of routing; cloud-first with more limited
  self-hosting than LiteLLM, though it open-sourced its core gateway (Apache 2.0) in
  2026.
- **OpenRouter** — the fastest path to a huge model catalog (300+ models) with zero
  setup; a hosted aggregator rather than something you run yourself, with a pricing
  markup baked in.
- **Kong AI Gateway** — LLM routing built into Kong's established API-management
  platform; the natural choice if your org already runs a Kong API mesh, with strong
  enterprise SSO and PII-redaction plugins.
- **Cloudflare AI Gateway** — an edge-based proxy offering caching, rate-limiting, and
  analytics with essentially zero infrastructure to operate.

### Decision shortcut
Self-hosted + cost-focused → **LiteLLM**. Need built-in guardrails/governance without
running your own infra → **Portkey**. Want the widest model catalog with zero setup →
**OpenRouter**. Already operating Kong → **Kong AI Gateway**.


## 8. Guardrails & Content Moderation

### Why it matters
A model that responds accurately to a benign request can still produce unsafe,
policy-violating, or brand-damaging output when pushed — and an agent with tool access
that acts on a manipulated instruction can cause real-world harm, not just a bad
message. Guardrails are the layer that catches this *regardless of whether the root
cause was a malicious user, a poisoned document, or an unlucky generation.*

### What it covers (a layered view)
Production LLM safety is generally implemented as several distinct layers, each
addressing a different failure mode rather than one blanket "safety toggle":
1. **Input classification** — flagging unsafe or adversarial prompts before they
   reach the model.
2. **Dialogue/policy enforcement** — constraining what topics or actions the model is
   allowed to engage with mid-conversation.
3. **Retrieval-stage filtering** — screening content pulled into context via RAG or
   tool results for injected instructions (this is the layer that catches *indirect*
   injection, which input-only classifiers miss).
4. **Output filtering** — checking the model's own response for unsafe content, PII
   leakage, or system-prompt disclosure before it's returned to the user.
5. **Structured-output validation** — enforcing that responses conform to an expected
   schema (useful well beyond safety, e.g. guaranteeing valid JSON for downstream
   parsing).
6. **Tool/action-level gating** — a hard permission boundary on what an agent is
   actually allowed to execute, independent of what the model *says* it intends to do.

### How to achieve it — the tools
- **Llama Guard / Prompt Guard (Meta)** — lightweight, purpose-trained classifier
  models for detecting unsafe prompts and jailbreak/injection attempts at the input
  layer; commonly cited as outperforming general-purpose models on safety
  classification specifically.
- **NeMo Guardrails (NVIDIA)** — a programmable, dialogue-flow-aware framework (using
  its own Colang language) covering input, dialogue, retrieval, and output rails in
  one configurable system; a strong fit when an agent needs clearly defined topical
  boundaries.
- **LLM Guard** — broad out-of-the-box coverage (prompt injection detection, PII
  anonymization/masking) aimed at production apps that need protection without heavy
  custom rule-writing.
- **Guardrails AI** — centers on structured-output validation via the RAIL
  specification; the right tool specifically when an LLM's response must conform to a
  strict schema (forms, API payloads, reports).
- **Rebuff, Lakera Guard** — dedicated prompt-injection-focused layers, often used to
  add a second line of defense on top of a broader framework like LLM Guard or NeMo.

### Practical note
Most production systems combine **2–3 of these tools** rather than relying on a single
one — e.g. LLM Guard for broad input/PII coverage, NeMo Guardrails for dialogue policy,
and explicit tool-permission scoping for anything with agentic capability. No single
filter is sufficient on its own, especially against indirect injection.


## 9. Licensing Considerations

### Why it matters
Unlike a proprietary API where usage terms are set entirely by the provider, an
open-weight model comes with a **license attached to the weights themselves** — and
that license determines what you're actually allowed to do with a deployment, not just
what's technically possible. Deploying a model commercially without checking its
license is a real legal exposure, not a formality — restrictions can cover
commercial use thresholds, redistribution, or the use of model outputs to train other
models.

### What varies between licenses
- **Commercial-use permissions** — some licenses are fully permissive, others restrict
  commercial use above a certain scale (e.g. monthly active user thresholds) or
  require a separate commercial agreement.
- **Redistribution terms** — whether you can redistribute the weights themselves,
  fine-tuned derivatives, or only your own hosted service built on top.
- **Output-usage restrictions** — some licenses restrict using a model's *outputs* to
  train a competing model.
- **Attribution and naming requirements** — some require crediting the base model or
  restrict how a derivative can be branded/named.

### The major license families to know
- **Permissive open-source (Apache 2.0, MIT)** — minimal restrictions, commercial use
  generally unrestricted; common for models released by research labs or smaller
  organizations wanting broad adoption.
- **Community/custom licenses (e.g. Meta's Llama Community License)** — free to use
  and modify, but with specific conditions attached (historically including
  scale-based commercial thresholds and restrictions on using outputs to improve
  competing LLMs) — always worth reading the current version rather than assuming
  terms from an earlier model generation still apply.
- **OpenRAIL / RAIL-family licenses** — "Responsible AI Licenses" that combine open
  access with behavioral-use restrictions (e.g. prohibiting certain harmful use
  cases) baked directly into the license terms rather than left to a separate
  usage policy.
- **Fully closed/proprietary weights** — no redistribution rights at all; access is
  typically API-only rather than a downloadable model.

### How to actually manage this
There isn't a single tool that solves licensing — it's a process, not infrastructure:
- Check the **specific license file** shipped with each model on its model card
  (Hugging Face model cards link the exact license) before deployment, rather than
  assuming based on the model family's reputation.
- Track which license applies to which deployed model as part of your model registry
  metadata (ties into Section 4's MLflow Model Registry) so it's auditable later.
- Re-check licensing on every model **version** update — organizations have changed
  license terms between releases of the same model family.


## 10. How It All Fits Together

These 9 pieces aren't independent checkboxes — they interlock into a single request's
lifecycle:

```text
                                Request arrives
                                       │
                                       ▼
                    ┌───────────────────────────────────┐
                    │   Gateway (Section 7)              │
                    │   auth, rate limits, routing,       │
                    │   budget tracking                   │
                    └──────────────┬──────────────────────┘
                                    ▼
                    ┌───────────────────────────────────┐
                    │   Guardrails — input layer          │
                    │   (Sections 2 & 8)                  │
                    │   injection/jailbreak detection      │
                    └──────────────┬──────────────────────┘
                                    ▼
                    ┌───────────────────────────────────┐
                    │   Cache check (Section 6)           │
                    │   semantic + prefix/KV cache         │
                    └──────────────┬──────────────────────┘
                                    ▼  (cache miss)
                    ┌───────────────────────────────────┐
                    │   Serving engine (earlier notebook) │
                    │   vLLM / SGLang / llama.cpp / etc.  │
                    └──────────────┬──────────────────────┘
                                    ▼
                    ┌───────────────────────────────────┐
                    │   Guardrails — output layer          │
                    │   (Section 8): PII, unsafe content   │
                    └──────────────┬──────────────────────┘
                                    ▼
                    ┌───────────────────────────────────┐
                    │   Observability (Section 1)          │
                    │   trace, cost, latency logged        │
                    └──────────────┬──────────────────────┘
                                    ▼
                              Response returned
```

And running **around** this per-request path, continuously:
- **Evaluation (Section 5)** gates every new prompt/model version before it's allowed
  into this pipeline at all.
- **CI/CD (Section 4)** governs how a new version gets promoted through canary →
  production, and how quickly a bad version can be rolled back.
- **Cost optimization (Section 3)** and **licensing (Section 9)** are ongoing
  governance concerns that sit above any single request.

**The core takeaway:** "getting a model to respond" is the easy 20% of LLM deployment.
The other 80% — the 9 areas in this notebook — is what separates a demo from a system
you can actually trust in production.
